In [1]:
# =========================================================
# PyTorch Robust BiLSTM Complete Training Script
# =========================================================

import random
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from torch.utils.data import Dataset, DataLoader

from torch.nn.utils.rnn import (
    pad_sequence,
    pack_padded_sequence,
    pad_packed_sequence
)

from torch.nn.utils import clip_grad_norm_

# =========================================================
# Reproducibility
# =========================================================

torch.manual_seed(42)
random.seed(42)

# =========================================================
# Device
# =========================================================

device = torch.device("cpu")

# =========================================================
# Synthetic Dataset
# =========================================================

class ToySequenceDataset(Dataset):

    def __init__(
        self,
        n_samples=500,
        vocab_size=50,
        min_len=5,
        max_len=20
    ):

        self.samples = []

        for _ in range(n_samples):

            length = random.randint(min_len, max_len)

            seq = torch.randint(
                1,
                vocab_size,
                (length,),
                dtype=torch.long
            )

            # Binary classification
            # Even sum -> class 1
            # Odd sum -> class 0

            label = int(seq.sum().item() % 2 == 0)

            self.samples.append((seq, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

# =========================================================
# Collate Function
# =========================================================

def collate_fn(batch, pad_value=0):

    sequences, labels = zip(*batch)

    lengths = torch.tensor(
        [len(seq) for seq in sequences],
        dtype=torch.long
    )

    padded_sequences = pad_sequence(
        sequences,
        batch_first=True,
        padding_value=pad_value
    )

    labels = torch.tensor(
        labels,
        dtype=torch.long
    )

    return padded_sequences, lengths, labels

# =========================================================
# Robust BiLSTM Model
# =========================================================

class RobustBiLSTM(nn.Module):

    def __init__(
        self,
        vocab_size,
        embed_dim,
        hidden_dim,
        num_layers,
        num_classes,
        dropout=0.3,
        pad_idx=0
    ):

        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=pad_idx
        )

        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0
        )

        self.dropout = nn.Dropout(dropout)

        self.classifier = nn.Linear(
            hidden_dim * 2,
            num_classes
        )

    def forward(self, x, lengths, hidden=None):

        x = self.embedding(x)

        x = self.dropout(x)

        lengths_cpu = lengths.to("cpu")

        packed = pack_padded_sequence(
            x,
            lengths_cpu,
            batch_first=True,
            enforce_sorted=False
        )

        packed_out, (h_n, c_n) = self.lstm(
            packed,
            hidden
        )

        unpacked_out, _ = pad_packed_sequence(
            packed_out,
            batch_first=True
        )

        forward_last = h_n[-2]

        backward_last = h_n[-1]

        features = torch.cat(
            [forward_last, backward_last],
            dim=1
        )

        features = self.dropout(features)

        logits = self.classifier(features)

        return logits, features, (h_n, c_n), unpacked_out

# =========================================================
# Orthogonal Initialization
# =========================================================

def init_lstm_orthogonal(model):

    for name, param in model.named_parameters():

        if "weight_ih" in name:

            nn.init.xavier_uniform_(param.data)

        elif "weight_hh" in name:

            hidden_dim = param.shape[1]

            for start in range(
                0,
                param.shape[0],
                hidden_dim
            ):

                nn.init.orthogonal_(
                    param.data[
                        start:start + hidden_dim
                    ]
                )

        elif "bias" in name:

            nn.init.zeros_(param.data)

            hidden_dim = param.shape[0] // 4

            # Forget gate bias = 1

            param.data[
                hidden_dim:2 * hidden_dim
            ].fill_(1.0)

# =========================================================
# Hidden-State Detach
# =========================================================

def detach_state(state):

    if state is None:
        return None

    if isinstance(state, tuple):

        return tuple(
            s.detach() for s in state
        )

    return state.detach()

# =========================================================
# Hyperparameters
# =========================================================

vocab_size = 50
embed_dim = 32
hidden_dim = 64
num_layers = 2
num_classes = 2

batch_size = 16

epochs = 10

learning_rate = 1e-3

max_norm = 1.0

# =========================================================
# Dataset and Loader
# =========================================================

train_dataset = ToySequenceDataset(
    n_samples=500,
    vocab_size=vocab_size
)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=collate_fn
)

# =========================================================
# Model
# =========================================================

model = RobustBiLSTM(
    vocab_size=vocab_size,
    embed_dim=embed_dim,
    hidden_dim=hidden_dim,
    num_layers=num_layers,
    num_classes=num_classes,
    dropout=0.3
).to(device)

# =========================================================
# Initialization
# =========================================================

init_lstm_orthogonal(model)

# =========================================================
# Loss and Optimizer
# =========================================================

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=learning_rate
)

# =========================================================
# Training History
# =========================================================

history_loss = []

history_acc = []

# =========================================================
# Training Loop
# =========================================================

print("\nStarting Training...\n")

for epoch in range(epochs):

    model.train()

    total_loss = 0.0

    total_correct = 0

    total_samples = 0

    hidden = None

    for inputs, lengths, labels in train_loader:

        inputs = inputs.to(device)

        lengths = lengths.to(device)

        labels = labels.to(device)

        optimizer.zero_grad()

        hidden = detach_state(hidden)

        logits, features, hidden, unpacked = model(
            inputs,
            lengths,
            hidden
        )

        loss = criterion(
            logits,
            labels
        )

        loss.backward()

        # Gradient Clipping

        clip_grad_norm_(
            model.parameters(),
            max_norm=max_norm
        )

        optimizer.step()

        total_loss += (
            loss.item() * labels.size(0)
        )

        predictions = logits.argmax(dim=1)

        total_correct += (
            predictions == labels
        ).sum().item()

        total_samples += labels.size(0)

        hidden = None

    epoch_loss = total_loss / total_samples

    epoch_acc = total_correct / total_samples

    history_loss.append(epoch_loss)

    history_acc.append(epoch_acc)

    print(
        f"Epoch {epoch+1:02d} | "
        f"Loss: {epoch_loss:.4f} | "
        f"Accuracy: {epoch_acc:.4f}"
    )

# =========================================================
# Save Console-like Output Figure
# =========================================================

plt.figure(figsize=(8,4))

plt.plot(
    range(1, epochs+1),
    history_loss,
    marker='o'
)

plt.xlabel("Epoch")

plt.ylabel("Loss")

plt.title("Training Loss")

plt.grid(True)

plt.savefig("k4.png")

plt.close()

# =========================================================
# Accuracy Plot
# =========================================================

plt.figure(figsize=(8,4))

plt.plot(
    range(1, epochs+1),
    history_acc,
    marker='o'
)

plt.xlabel("Epoch")

plt.ylabel("Accuracy")

plt.title("Training Accuracy")

plt.grid(True)

plt.savefig("k5.png")

plt.close()

# =========================================================
# Example Batch Visualization
# =========================================================

example_batch = next(iter(train_loader))

inputs, lengths, labels = example_batch

plt.figure(figsize=(10,5))

plt.imshow(inputs.numpy(), aspect='auto')

plt.colorbar()

plt.title("Padded Input Sequences")

plt.xlabel("Sequence Position")

plt.ylabel("Batch Sample")

plt.savefig("k1.png")

plt.close()

# =========================================================
# Feature Visualization
# =========================================================

model.eval()

with torch.no_grad():

    inputs = inputs.to(device)

    lengths = lengths.to(device)

    logits, features, hidden, unpacked = model(
        inputs,
        lengths
    )

feature_matrix = features.cpu().numpy()

plt.figure(figsize=(10,5))

plt.imshow(feature_matrix, aspect='auto')

plt.colorbar()

plt.title("BiLSTM Feature Representation")

plt.xlabel("Feature Dimension")

plt.ylabel("Batch Sample")

plt.savefig("k2.png")

plt.close()

# =========================================================
# Logits Visualization
# =========================================================

logit_matrix = logits.cpu().numpy()

plt.figure(figsize=(8,5))

plt.imshow(logit_matrix, aspect='auto')

plt.colorbar()

plt.title("Classification Logits")

plt.xlabel("Class")

plt.ylabel("Batch Sample")

plt.savefig("k3.png")

plt.close()

# =========================================================
# Console Output Figure
# =========================================================

plt.figure(figsize=(10,5))

text = ""

for i in range(epochs):

    line = (
        f"Epoch {i+1:02d} | "
        f"Loss: {history_loss[i]:.4f} | "
        f"Accuracy: {history_acc[i]:.4f}\n"
    )

    text += line

plt.text(
    0.01,
    0.95,
    text,
    fontsize=12,
    verticalalignment='top',
    family='monospace'
)

plt.axis('off')

plt.title("Training Console Output")

plt.savefig("k6.png")

plt.close()

# =========================================================
# Final Output
# =========================================================

print("\nTraining Finished Successfully.\n")

print("Generated Images:")

print("k1.png -> Padded sequences")
print("k2.png -> BiLSTM feature representation")
print("k3.png -> Classification logits")
print("k4.png -> Training loss curve")
print("k5.png -> Training accuracy curve")
print("k6.png -> Console training output")


Starting Training...

Epoch 01 | Loss: 0.7027 | Accuracy: 0.5120
Epoch 02 | Loss: 0.6829 | Accuracy: 0.5500
Epoch 03 | Loss: 0.6731 | Accuracy: 0.5680
Epoch 04 | Loss: 0.6538 | Accuracy: 0.6280
Epoch 05 | Loss: 0.6108 | Accuracy: 0.6600
Epoch 06 | Loss: 0.5915 | Accuracy: 0.6600
Epoch 07 | Loss: 0.6038 | Accuracy: 0.6660
Epoch 08 | Loss: 0.5685 | Accuracy: 0.7020
Epoch 09 | Loss: 0.5232 | Accuracy: 0.7260
Epoch 10 | Loss: 0.5385 | Accuracy: 0.7520

Training Finished Successfully.

Generated Images:
k1.png -> Padded sequences
k2.png -> BiLSTM feature representation
k3.png -> Classification logits
k4.png -> Training loss curve
k5.png -> Training accuracy curve
k6.png -> Console training output
